# Einstein examples 2 and 3 — practice notebook

**TLDR:** Implement image dimming and pixel mixing from the assignment PDF.
The inputs are ready; the operations are yours to write.

Last edited: 2026-10-24 11:34:41 PDT

Source: [cs336_assignment1_basics.pdf](../cs336_assignment1_basics.pdf), printed pages 15–16, `einstein_example2` and `einstein_example3`.
The exercises below restate those examples without their solutions.

Select the Python environment at `assignment1-basics/.venv/bin/python` as your notebook kernel.
If your editor asks to install `ipykernel` in that environment, accept it.
Run the setup cells, then fill in the blank implementation cells.


In [2]:
import sys
print(sys.executable)

/Users/jacquesattinger/code/cs336/assignment1-basics/.venv/bin/python


In [3]:
# Last edited: 2026-10-24 11:34:41 PDT
import torch
from einops import rearrange, einsum

# Repeatable random inputs, with small tensors that are easy to inspect.
torch.manual_seed(0)
torch.set_printoptions(precision=3, linewidth=100)


## Example 2: dim an image batch

For every image, create one dimmed version for each scaling factor.
A factor of 0 makes every value zero; a factor of 1 keeps the input unchanged.
Each factor multiplies every pixel and channel of its image.

| Name | Shape | Meaning |
| --- | --- | --- |
| `images` | `(batch, height, width, channel)` | Input images |
| `dim_by` | `(num_dims,)` | Scaling factors |
| `dimmed_images` | `(batch, num_dims, height, width, channel)` | Desired output |

The PDF uses `batch=64`, `height=width=128`, `channel=3`, and `num_dims=10`.
Start with the smaller sizes below.


In [4]:
batch, height, width, channel = 2, 4, 5, 3
num_dims = 10
images = torch.randn(batch, height, width, channel)
dim_by = torch.linspace(start=0.0, end=1.0, steps=num_dims)

print("images:", images.shape)
print("scaling factors:", dim_by)


images: torch.Size([2, 4, 5, 3])
scaling factors: tensor([0.000, 0.111, 0.222, 0.333, 0.444, 0.556, 0.667, 0.778, 0.889, 1.000])


### 2A. Implement with `rearrange` and multiplication

Reshape the two inputs with `rearrange`, then multiply them to produce `dimmed_images`.
PyTorch can repeat an axis of length 1 to match another tensor during multiplication; this is called broadcasting.
Before coding, write down the two shapes you want to multiply.


In [5]:
# Last edited: 2026-10-24 19:02 UTC
# Target shapes:
#   images: (batch, 1, height, width, channel)
#   dim_by: (1, num_dims, 1, 1, 1)
images_for_broadcast = rearrange(
    images, "batch height width channel -> batch 1 height width channel"
)
dim_by_for_broadcast = rearrange(
    dim_by, "num_dims -> 1 num_dims 1 1 1"
)
dimmed_images = images_for_broadcast * dim_by_for_broadcast


### 2B. Implement with `einsum`

Use the original `images` and `dim_by` to produce `dimmed_images_einsum` in one `einsum` call.
Write the pattern yourself.


In [6]:
# TODO: Compute dimmed_images_einsum. Shape of image is 4, 5, 3
print(images[0, :, :, 0]) # prints the first image, but only its red color values
Y = einsum(images, dim_by, "batch height width channel, dim_value -> batch dim_value height width channel")
print(f'Y shape is {Y.shape}')
print(Y[0, 9, :, :, 0])


tensor([[-1.126, -0.434, -0.316, -1.263,  0.120],
        [-0.247,  0.567, -1.555,  0.750,  0.183],
        [ 0.946,  0.032,  0.440,  0.441, -0.290],
        [ 2.302, -0.673,  0.178,  0.543,  0.744]])
Y shape is torch.Size([2, 10, 4, 5, 3])
tensor([[-1.126, -0.434, -0.316, -1.263,  0.120],
        [-0.247,  0.567, -1.555,  0.750,  0.183],
        [ 0.946,  0.032,  0.440,  0.441, -0.290],
        [ 2.302, -0.673,  0.178,  0.543,  0.744]])


### Check and experiment

- Check the output shape against the table.
- Check that the first scaling factor produces zeros and the last reproduces `images`.
- Compare both implementations with `torch.testing.assert_close`.
- Pick one image, scaling factor, pixel, and channel; check that value by hand.
- Try one scaling factor, a factor above 1, and a negative factor.
- Use distinct height and width sizes to help catch swapped axes.

The inputs are random numbers, so they can contain negative values.
They are useful for tensor practice, but are not display-ready photographs.


In [7]:
# Your checks and experiments:


## Example 3: mix pixels within each channel

For each image and channel, apply the same linear transformation to all spatial positions.
Each output pixel is a weighted sum of the input pixels in that channel.
Do not mix different images or different channels.

| Name | Shape | Meaning |
| --- | --- | --- |
| `channels_last` | `(batch, height, width, channel)` | Input images |
| `B` | `(height * width, height * width)` | Pixel mixing weights |
| `channels_last_transformed` | Same as `channels_last` | Desired output |

The first axis of `B` selects the output pixel.
The second axis selects the input pixel.
Flatten spatial positions in row order: pixel index = row × width + column.

The PDF uses `batch=64`, `height=width=32`, and `channel=3`.
This exercise starts smaller and uses different height and width sizes.


In [63]:
batch, height, width, channel = 2, 3, 4, 3
num_pixels = height * width
channels_last = torch.randn(batch, height, width, channel)
B = torch.randn(num_pixels, num_pixels)

print("channels_last:", channels_last.shape)
print("B:", B.shape)


channels_last: torch.Size([2, 3, 4, 3])
B: torch.Size([12, 12])


### 3A. Implement with plain PyTorch

Use reshaping, transposing, and matrix multiplication to compute `channels_last_transformed_torch`.
Write the shape beside each intermediate tensor.
Keep the output layout the same as the input layout.


In [64]:
# TODO: Implement the pixel transformation with plain PyTorch.
channels_last_compressed = channels_last.reshape(batch, height * width, channel)
print("channels_last:", channels_last_compressed.shape)
print(channels_last_compressed[0][0])


# Perform the matrix multiplication
R = B @ channels_last_compressed
print("R:", R.shape)
print("channels_last_compressed after multiplication:", channels_last_compressed[0][0])
R = R.reshape(batch, height, width, channel)
print("R:", R.shape)
print(R[0][0][0])

channels_last: torch.Size([2, 12, 3])
tensor([-0.552, -0.464, -0.412])
R: torch.Size([2, 12, 3])
channels_last_compressed after multiplication: tensor([-0.552, -0.464, -0.412])
R: torch.Size([2, 3, 4, 3])
tensor([-2.842,  0.549, -4.876])


### 3B. Implement with `rearrange` and `einsum`

Compute `channels_last_transformed` from the original input and `B`.
Use `rearrange` to prepare the spatial positions, `einsum` to apply the weights, and `rearrange` to restore the image layout.
Choose the dimension names and patterns yourself.


In [68]:
# TODO: Implement the pixel transformation with einops.
flat = rearrange(channels_last, "b h w c -> b (h w) c")
print("flat:", flat.shape)
print(flat[0][0])
result = einsum(flat, B, "b y c, x y -> b x c")
print("result after multiplication:", result[0][0])

print("result:", result.shape)
result = rearrange(result, "b (h w) c -> b h w c", h = height, w = width)
print("result:", result.shape)
print(result[0][0][0])




flat: torch.Size([2, 12, 3])
tensor([-0.552, -0.464, -0.412])
result after multiplication: tensor([-2.842,  0.549, -4.876])
result: torch.Size([2, 12, 3])
result: torch.Size([2, 3, 4, 3])
tensor([-2.842,  0.549, -4.876])


### Check and experiment

- Check that the output has the same shape as `channels_last`.
- Compare the PyTorch and einops outputs with `torch.testing.assert_close`.
- Replace `B` with an identity matrix; the output should match the input.
- Replace `B` with zeros; the output should be zero.
- Change just one input channel; the other output channels should stay unchanged.
- Change just one image; the other output images should stay unchanged.
- For a 2 × 2 image with one channel, calculate one output pixel by hand.
- Try weights that swap two pixels, then inspect where those pixels move.

Use distinct weights when you check the direction of the transformation.
An identity matrix alone cannot reveal a mistaken transpose.


In [69]:
# Your checks and experiments:
same = torch.allclose(R, result)
print(same)

True


### Optional: one call with `einx.dot`

The PDF also shows a single-call version using `einx.dot`.
If you want another challenge, import `einx` and express the same transformation in one call.
Compare its result with your previous implementations.


In [ ]:
# Optional implementation and experiments:
